In [1]:
import os

In [2]:
%pwd


'/mnt/d/resume_projects/flight_fare_prediction/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/mnt/d/resume_projects/flight_fare_prediction'

In [5]:
from dataclasses import dataclass
from pathlib import Path

In [6]:
@dataclass
class DataTransformationConfig:
    root_dir: Path
    transformed_train_file_name: Path
    transformed_test_file_name: Path
    preprocessor_object_file_name: Path
    target_column: str
    numerical_columns: list
    categorical_columns: list
    scaler: str
    encoder: str
    

In [7]:
#artifact entity
@dataclass
class DataTransformationArtifact:
    transformed_train_file_name: Path
    transformed_test_file_name: Path
    preprocessor_object_file_name: Path

In [8]:
from src.flight_price_prediction.constants import *
from src.flight_price_prediction.entity.config_entity import DataTransformationConfig
from src.flight_price_prediction.utils.common import *
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [9]:
#updating configuration manager
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        params = self.params.data_transformation

        transformed_train_file_name = Path(config.transformed_train_file_name)
        transformed_test_file_name = Path(config.transformed_test_file_name)
        preprocessor_object_file_name = Path(config.preprocessor_object_file_name)
        target_column=params.target_column
        numerical_columns=list(params.numerical_columns)
        categorical_columns=list(params.categorical_columns)
        scaler=StandardScaler()
        encoder=OneHotEncoder(handle_unknown='ignore', sparse_output=False

        )


        create_directories([Path(config.root_dir),transformed_train_file_name.parent,
                            transformed_test_file_name.parent,
                            preprocessor_object_file_name.parent])
        
        return DataTransformationConfig(
            root_dir = Path(config.root_dir),
            transformed_train_file_name = transformed_train_file_name,
            transformed_test_file_name = transformed_test_file_name,
            preprocessor_object_file_name = preprocessor_object_file_name,
            target_column= target_column,
            numerical_columns=numerical_columns,
            categorical_columns=categorical_columns,
            scaler=scaler,
            encoder=encoder
        
        )
        



In [10]:
# update component
from src.flight_price_prediction.entity.config_entity import DataTransformationConfig
from  src.flight_price_prediction.entity.artifact_entity import FeatureEngineeringArtifact,DataTransformationArtifact
from src.flight_price_prediction.exception.exception import CustomException
from src.flight_price_prediction.logging.logger import logging
from src.flight_price_prediction.utils.common import save_bin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


In [11]:
class DataTranformation:
    def ___init__(self, 
                  data_transformation_config: DataTransformationConfig,
                  feature_engineering_artifact: FeatureEngineeringArtifact):
        try:
            self.config = data_transformation_config
            self.feature_engineering_artifact = feature_engineering_artifact
        except Exception as e:
            raise CustomException(e, sys)
    
    @staticmethod
    def read_data(file_path) -> pd.DataFrame:
        try:
            return pd.read_csv(file_path)
        except Exception as e:
            raise CustomException(e, sys)



    def _build_transformer(self) -> Pipeline:
        numeric_transformer = self.config.scaler
        categorical_transformer = self.config.encoder
        try:
            preprocessor = ColumnTransformer(
                transformers=[
                    ('numeric' , numeric_transformer, self.config.numerical_columns),
                    ('categorical', categorical_transformer, self.config.categorical_columns),
                ],
                remainder ='drop',
            )

            pipeline = Pipeline(steps = [('preprocessor', preprocessor)])
            return pipeline
        except Exception as e:
            raise CustomException(e, sys)
        
    def initiate_data_transformation(self) -> DataTransformationArtifact:
        try:
            logging.info('Starting data transformation on engineered datasets')
            
            train_df = DataTranformation.read_data(self.feature_engineering_artifact.engineered_train_file_name)
            test_df = DataTranformation.read_data(self.feature_engineering_artifact.engineered_test_file_name)

            target_column = self.config.target_column

            x_train = train_df.drop(columns =[target_column])
            y_train = train_df[target_column].values
            x_test = test_df.drop(columns = [target_column])
            y_test = test_df[target_column].values

            transformer = self._build_transformer()
            x_train_transformed = transformer.fit_transform(x_train)
            x_test_transformed = transformer.transform(x_test)

            train_array = np.c_[x_train_transformed, y_train]
            test_array = np.c_[x_test_transformed, y_test]

            save_bin(train_array , self.config.transformed_train_file_name)
            save_bin(test_array, self.config.transformed_test_file_name)
            joblib.dump(transformer, self.config.preprocessor_object_file_name)

            logging.info(f'Transformed train data saved to {self.config.transformed_train_file_name}')
            logging.info(f'Transformed test data saved to {self.config.transformed_test_file_name}')
            logging.info(f'Preprocessor object saved to {self.config.preprocessor_object_file_name}')

            return DataTransformationArtifact(
                transformed_train_file_name=self.config.transformed_train_file_name,
                transformed_test_file_name=self.config.transformed_test_file_name,
                preprocessor_object_file_path=self.config.preprocessor_object_file_name,
            )
        except Exception as e:
            raise CustomException(e, sys)












            


In [12]:
import sys
from src.flight_price_prediction.config.configuration import ConfigurationManager
from src.flight_price_prediction.components.data_transformation import DataTransformation
from src.flight_price_prediction.entity.artifact_entity import FeatureEngineeringArtifact, DataTransformationArtifact
from src.flight_price_prediction.exception.exception import CustomException
from src.flight_price_prediction.logging.logger import logging


In [13]:
STAGE_NAME = "Data Transformation Stage"

class DataTransformationTrainingPipeline:
    def __init__(self, config: ConfigurationManager, feature_engineering_artifact: FeatureEngineeringArtifact):
        try:
            self.config = config
            self.feature_engineering_artifact = feature_engineering_artifact
        except Exception as e:
            raise CustomException(e, sys)

    def initiate_data_transformation(self) -> DataTransformationArtifact:
        try:
            logging.info(f'>>>> stage {STAGE_NAME} started <<<<')
            data_transformation_config = self.config.get_data_transformation_config()
            data_transformation = DataTransformation(
                data_transformation_config=data_transformation_config,
                feature_engineering_artifact=self.feature_engineering_artifact,
            )
            artifact = data_transformation.initiate_data_transformation()
            logging.info(f'>>>> stage {STAGE_NAME} completed <<<<')
            return artifact
        except Exception as e:
            raise CustomException(e, sys)

if __name__ == '__main__':
    try:
        logging.info(f'>>>> stage {STAGE_NAME} started <<<<')
        config = ConfigurationManager()
        from src.flight_price_prediction.pipeline.feature_engineering_pipeline import FeatureEngineeringTrainingPipeline
        from src.flight_price_prediction.pipeline.data_ingestion_pipeline import DataIngestionTrainingPipeline
        from src.flight_price_prediction.pipeline.data_validation_pipeline import DataValidationTrainingPipeline

        ingestion_pipeline = DataIngestionTrainingPipeline(config=config)
        ingestion_artifact = ingestion_pipeline.initiate_data_ingestion()
        validation_pipeline = DataValidationTrainingPipeline(config=config, data_ingestion_artifact=ingestion_artifact)
        validation_artifact = validation_pipeline.initiate_data_validation()
        feature_engineering_pipeline = FeatureEngineeringTrainingPipeline(config=config, data_validation_artifact=validation_artifact)
        fe_artifact = feature_engineering_pipeline.initiate_feature_engineering()
        pipeline = DataTransformationTrainingPipeline(config=config, feature_engineering_artifact=fe_artifact)
        pipeline.initiate_data_transformation()
        logging.info(f'>>>> stage {STAGE_NAME} completed <<<<')
    except Exception as e:
        raise CustomException(e, sys)

[2026-06-10 07:52:33,726: INFO: 2702210898: >>>> stage Data Transformation Stage started <<<<]
[2026-06-10 07:52:33,739: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/config/config.yaml loaded succesfully ]
[2026-06-10 07:52:33,753: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/params/params.yaml loaded succesfully ]
[2026-06-10 07:52:33,763: INFO: common: yaml file: /mnt/d/resume_projects/flight_fare_prediction/schema/schema.yaml loaded succesfully ]
[2026-06-10 07:52:33,768: INFO: common: created directory at: artifacts]
[2026-06-10 07:52:35,011: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-10 07:52:35,572: INFO: data_ingestion: Connecting to MongoDB at: mongodb+srv://p...]
[2026-06-10 07:52:36,423: INFO: data_ingestion: Successfully connected to MongoDB.]
[2026-06-10 07:52:56,784: INFO: common: Data saved to: artifacts/data_ingestion/feature_store/flight_fare.csv]
[2026-06-10 07:52:56,786: INFO: data_ingesti